In [39]:
import pandas as pd
import numpy as np

customers_data = {
    "customer_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111],
    "customer_name": [
        "Asha Rao", "Ravi Kumar", "Meena Devi", "Kiran Reddy",
        "Priya Nair", "Arjun Varma", "Sneha Paul", "Rahul Das",
        "Divya Patel", "Nikhil Jain", "Anil Kumar"
    ],
    "city": [
        "Hyderabad", "Vijayawada", "Visakhapatnam", "Bengaluru",
        "Chennai", "Mumbai", "Pune", "Delhi", "Ahmedabad",
        "Kolkata", "Guntur"
    ],
    "region": [
        "South", "South", "South", "South", "South",
        "West", "West", "North", "West", "East", "South"
    ]
}

customers_df = pd.DataFrame(customers_data)
customers_df.to_csv("customers.csv", index=False)

print("Customers CSV created successfully!")
display(customers_df)

Customers CSV created successfully!


,customer_id,customer_name,city,region
0,101,Asha Rao,Hyderabad,South
1,102,Ravi Kumar,Vijayawada,South
2,103,Meena Devi,Visakhapatnam,South
3,104,Kiran Reddy,Bengaluru,South
4,105,Priya Nair,Chennai,South
5,106,Arjun Varma,Mumbai,West
6,107,Sneha Paul,Pune,West
7,108,Rahul Das,Delhi,North
8,109,Divya Patel,Ahmedabad,West
9,110,Nikhil Jain,Kolkata,East


In [40]:
orders_data = {
    "order_id": [
        "O-2001", "O-2002", "O-2003", "O-2004", "O-2005",
        "O-2006", "O-2007", "O-2008", "O-2009", "O-2010",
        "O-2011", "O-2012", "O-2013", "O-2014", "O-2015",
        "O-2016", "O-2017", "O-2018", "O-2019", "O-2020"
    ],
    "customer_id": [
        101, 102, 103, 101, 104,
        105, 106, 103, 107, 108,
        101, 104, 109, 102, 105,
        103, 106, 108, 104, 110
    ],
    "order_amount": [
        12500, 2400, 8900, 3500, 15400,
        7200, 4500, 2100, 6800, 1750,
        5600, 4200, 9900, 3100, 2600,
        4700, 1250, 2800, 3600, 5300
    ]
}

orders_df = pd.DataFrame(orders_data)
orders_df.to_csv("orders.csv", index=False)

print("Orders CSV created successfully!")
display(orders_df.head())

Orders CSV created successfully!


,order_id,customer_id,order_amount
0,O-2001,101,12500
1,O-2002,102,2400
2,O-2003,103,8900
3,O-2004,101,3500
4,O-2005,104,15400


In [41]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

print("Customers shape:", customers.shape)
print("Orders shape:", orders.shape)
inner_merged = pd.merge(customers,orders,on="customer_id",how="inner")
print("Inner join shape:", inner_merged.shape)
left_merged = pd.merge(customers,orders,on="customer_id",how="left")
print("Left join shape:", left_merged.shape)

Customers shape: (11, 4)
Orders shape: (20, 3)
Inner join shape: (20, 6)
Left join shape: (21, 6)


In [42]:
non_converting_customers = left_merged[left_merged["order_id"].isna()][["customer_id", "customer_name", "city", "region"]].drop_duplicates()
print("Number of non-converting customers:",len(non_converting_customers))
display(non_converting_customers)

Number of non-converting customers: 1


,customer_id,customer_name,city,region
20,111,Anil Kumar,Guntur,South


In [43]:

customer_kpis = (inner_merged.groupby(["customer_id", "customer_name", "city", "region"],as_index=False).agg(
        total_spend=("order_amount", "sum"),
        order_count=("order_id", "nunique"),
        average_order_value=("order_amount", "mean")
    )
)
customer_kpis = customer_kpis[customer_kpis["order_count"] > 0].copy()
customer_kpis["average_order_value"] = (customer_kpis["average_order_value"].round(2))
customer_kpis = customer_kpis.sort_values("total_spend", ascending=False)
display(customer_kpis)

,customer_id,customer_name,city,region,total_spend,order_count,average_order_value
3,104,Kiran Reddy,Bengaluru,South,23200,3,7733.33
0,101,Asha Rao,Hyderabad,South,21600,3,7200.00
2,103,Meena Devi,Visakhapatnam,South,15700,3,5233.33
8,109,Divya Patel,Ahmedabad,West,9900,1,9900.00
4,105,Priya Nair,Chennai,South,9800,2,4900.00
6,107,Sneha Paul,Pune,West,6800,1,6800.00
5,106,Arjun Varma,Mumbai,West,5750,2,2875.00
1,102,Ravi Kumar,Vijayawada,South,5500,2,2750.00
9,110,Nikhil Jain,Kolkata,East,5300,1,5300.00
7,108,Rahul Das,Delhi,North,4550,2,2275.00


In [44]:
#CITY LEVEL ANALYSIS
city_kpis = (inner_merged.groupby(["city", "region"], as_index=False)
    .agg(
        total_spend=("order_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_buyers=("customer_id", "nunique"),
        average_order_value=("order_amount", "mean")
    )
)

city_kpis["average_order_value"] = (city_kpis["average_order_value"].round(2))
city_kpis = city_kpis.sort_values("total_spend", ascending=False)
display(city_kpis)

,city,region,total_spend,total_orders,unique_buyers,average_order_value
1,Bengaluru,South,23200,3,1,7733.33
4,Hyderabad,South,21600,3,1,7200.00
9,Visakhapatnam,South,15700,3,1,5233.33
0,Ahmedabad,West,9900,1,1,9900.00
2,Chennai,South,9800,2,1,4900.00
7,Pune,West,6800,1,1,6800.00
6,Mumbai,West,5750,2,1,2875.00
8,Vijayawada,South,5500,2,1,2750.00
5,Kolkata,East,5300,1,1,5300.00
3,Delhi,North,4550,2,1,2275.00


In [45]:
#REGION LEVEL ANALYSIS
region_kpis = (inner_merged.groupby("region", as_index=False)
    .agg(
        total_spend=("order_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_buyers=("customer_id", "nunique"),
        average_order_value=("order_amount", "mean")
    )
)

region_kpis["average_order_value"] = (region_kpis["average_order_value"].round(2))
region_kpis = region_kpis.sort_values("total_spend", ascending=False)
display(region_kpis)

,region,total_spend,total_orders,unique_buyers,average_order_value
2,South,75800,13,5,5830.77
3,West,22450,4,3,5612.50
0,East,5300,1,1,5300.00
1,North,4550,2,1,2275.00


In [46]:
total_revenue = orders["order_amount"].sum()
total_orders = orders["order_id"].nunique()
overall_aov = orders["order_amount"].mean()
customers_with_orders = orders["customer_id"].nunique()
customers_without_orders = (customers["customer_id"].nunique() - customers_with_orders)
print("ECOMMERCE BUSINESS SUMMARY")
print("-------------------------")
print("Total registered customers:", len(customers))
print("Customers with purchases:", customers_with_orders)
print("Customers without purchases:", customers_without_orders)
print("Total orders:", total_orders)
print("Total revenue:", f"{total_revenue:,.2f}")
print("Overall average order value:", f"{overall_aov:,.2f}")

ECOMMERCE BUSINESS SUMMARY
-------------------------
Total registered customers: 11
Customers with purchases: 10
Customers without purchases: 1
Total orders: 20
Total revenue: 108,100.00
Overall average order value: 5,405.00


In [47]:
top5_customers = customer_kpis.head(5)
assert customers["customer_id"].is_unique
assert orders["order_id"].is_unique
assert len(inner_merged) == len(orders)
assert len(non_converting_customers) == 1
assert len(top5_customers) == 5

print("All validation checks passed!")
display(top5_customers)

All validation checks passed!


,customer_id,customer_name,city,region,total_spend,order_count,average_order_value
3,104,Kiran Reddy,Bengaluru,South,23200,3,7733.33
0,101,Asha Rao,Hyderabad,South,21600,3,7200.00
2,103,Meena Devi,Visakhapatnam,South,15700,3,5233.33
8,109,Divya Patel,Ahmedabad,West,9900,1,9900.00
4,105,Priya Nair,Chennai,South,9800,2,4900.00
